# Claude로 구현하는 비동기 멀티에이전트 오케스트레이션

이 쿡북은 Claude Opus 4.8 시스템 카드의 멀티에이전트 결과 뒤에 있는 두 가지 오케스트레이션 패턴의 **형태**를 보여 줍니다. **고정 N-에이전트 팀**과 **비동기 서브에이전트**입니다. 여기에 도메인 작업은 없습니다. 오직 메시징과 서브에이전트 동작만 담아, 어떤 도구가 어떤 순서로 발동하는지 정확히 볼 수 있게 했습니다. 그 위에 여러분의 도구와 작업을 얹으면 됩니다.

전부 공개 [Anthropic Python SDK](https://github.com/anthropics/anthropic-sdk-python)와 `asyncio`로 동작하며, API 키만 있으면 대개 30초 안에 끝납니다.

## 준비

In [ ]:
%pip install -qU anthropic

In [4]:
import asyncio
import itertools
from collections import Counter, defaultdict

import anthropic

MODEL = "claude-opus-4-8"  # swap for the newest Claude model available to you
client = anthropic.AsyncAnthropic()  # reads ANTHROPIC_API_KEY from env

## 공통 구성 요소

### 메시지 허브

모든 에이전트는 받은 편지함 리스트와, 블로킹 대기를 위한 `asyncio.Event`를 갖습니다. `drain`은 받은 편지함을 비우고 구조화된 메시지를 반환하며, `render`는 도구 결과에 덧붙일 수 있도록 이를 텍스트로 정리합니다.

In [6]:
class Hub:
    def __init__(self):
        self.inbox: dict[str, list[dict]] = defaultdict(list)
        self.event: dict[str, asyncio.Event] = defaultdict(asyncio.Event)
        self.status: dict[str, str] = {}
        self._ids = itertools.count(1)

    def register(self, name: str):
        self.status[name] = "active"
        _ = self.inbox[name], self.event[name]

    def new_name(self, prefix="helper") -> str:
        n = f"{prefix}{next(self._ids)}"
        self.register(n)
        return n

    def post(self, sender: str, recipients: list[str], content: str) -> list[str]:
        delivered = []
        for rid in recipients:
            if rid in self.status:
                self.inbox[rid].append({"from": sender, "content": content})
                self.event[rid].set()
                delivered.append(rid)
        return delivered

    def drain(self, name: str) -> list[dict]:
        msgs, self.inbox[name] = self.inbox[name], []
        self.event[name] = asyncio.Event()
        return msgs

    @staticmethod
    def render(msgs: list[dict]) -> str:
        if not msgs:
            return ""
        body = "\n".join(
            f'<agent-message from="{m["from"]}">\n{m["content"]}\n</agent-message>' for m in msgs
        )
        return f"\n\n[Messages received while you were working:]\n{body}"

### 메시징 도구

두 패턴의 모든 에이전트가 이 두 도구를 갖습니다.

In [8]:
SEND_MESSAGE = {
    "name": "send_message",
    "description": (
        "Send a message to one or more other agents. It will appear appended to their next tool "
        "result. This is the ONLY way to reach other agents — plain text in your turn goes nowhere."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "recipient_ids": {"type": "array", "items": {"type": "string"}, "minItems": 1},
            "content": {"type": "string"},
        },
        "required": ["recipient_ids", "content"],
    },
}
WAIT_FOR_MESSAGE = {
    "name": "wait_for_message",
    "description": (
        "Block until another agent messages you. Note: messages also arrive automatically appended "
        "to the result of ANY other tool call, so only use this when you have nothing else to do."
    ),
    "input_schema": {"type": "object", "properties": {}},
}
BASE_TOOLS = [SEND_MESSAGE, WAIT_FOR_MESSAGE]

### 기본 에이전트 루프

함수 하나가 어떤 에이전트든 실행합니다. 표준 도구 사용 루프로, 허브를 상대로 `send_message` / `wait_for_message`를 처리하고, 추가 도구는 `extra_dispatch`로 보내며, **비운 받은 편지함을 마지막 도구 결과에 덧붙입니다**. 덕분에 에이전트는 폴링하지 않고 메시지가 인라인으로 도착합니다. 오케스트레이션이 펼쳐지는 모습을 볼 수 있도록 모든 도구 호출과 편지함 전달을 실시간으로 출력하고, 마지막에 요약 기록을 출력합니다.

In [10]:
TRACE: dict[str, list[str]] = defaultdict(list)


def _snip(s, n=60):
    s = str(s).replace("\n", " ")
    return s if len(s) <= n else s[:n] + "…"


async def run_agent(
    hub: Hub,
    name: str,
    system: str,
    first_user_turn: str,
    tools: list = None,
    extra_dispatch=None,
    max_turns: int = 20,
) -> str:
    tools = tools or BASE_TOOLS
    extra_dispatch = extra_dispatch or {}
    messages = [{"role": "user", "content": first_user_turn}]

    try:
        for _ in range(max_turns):
            resp = await client.messages.create(
                model=MODEL,
                max_tokens=2048,
                system=system,
                tools=tools,
                messages=messages,
            )
            messages.append({"role": "assistant", "content": resp.content})

            if resp.stop_reason == "end_turn":
                hub.status[name] = "done"
                return "".join(getattr(b, "text", "") for b in resp.content)
            if resp.stop_reason != "tool_use":
                raise RuntimeError(f"unexpected stop_reason: {resp.stop_reason}")

            results = []
            for block in resp.content:
                if block.type != "tool_use":
                    continue
                TRACE[name].append(block.name)
                if block.name == "send_message":
                    rids = block.input["recipient_ids"]
                    delivered = hub.post(name, rids, block.input["content"])
                    unknown = [r for r in rids if r not in delivered]
                    out = f"delivered to {delivered}" + (f"; unknown: {unknown}" if unknown else "")
                elif block.name == "wait_for_message":
                    hub.status[name] = "idling"
                    try:
                        await asyncio.wait_for(hub.event[name].wait(), timeout=60)
                        out = "woke: new messages"
                    except TimeoutError:
                        out = "woke: 60s timeout"
                    hub.status[name] = "active"
                elif block.name in extra_dispatch:
                    out = await extra_dispatch[block.name](block)
                else:
                    out = f"error: no dispatch for {block.name}"
                print(f"  [{name}] {block.name}({_snip(block.input)}) → {_snip(out)}")
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": out})

            inbox = hub.drain(name)
            for m in inbox:
                print(f"  [{name}] ← received from {m['from']}: {_snip(m['content'])}")
            if results:
                results[-1]["content"] += hub.render(inbox)  # ← the key line
            messages.append({"role": "user", "content": results})

        hub.status[name] = "done"
        return f"[{name} hit max_turns={max_turns}]"
    except Exception:
        hub.status[name] = "crashed"
        raise


def print_trace():
    for agent in sorted(TRACE):
        counts = Counter(TRACE[agent])
        print(f"  {agent}: " + ", ".join(f"{n}×{t}" for t, n in counts.most_common()))
    TRACE.clear()

---
## 1부 · 고정 N-에이전트 팀

에이전트 셋 — **리드** 하나와 **헬퍼** 둘 — 에게 각각 한 줄짜리 페르소나를 줍니다. 이들이 하는 일은 `send_message`로 서로에게 자기소개를 하는 것뿐이고, 리드가 한 문장 요약을 쓰고 실행을 끝냅니다. 도메인 도구는 없습니다. 순전히 메시징 경로만 보는 것입니다.

In [12]:
TEAM_SYSTEM = "You are {name}, one of 3 agents working together (peers: {peers})."

TASKS = {
    "lead": "You are the lead. Introduce yourself to the others. Once everyone has introduced "
    "themselves, finish with a one-sentence summary of the team.",
    "helper1": "You are a backend engineer named Ada. Introduce yourself to the others, then wait "
    "for their replies.",
    "helper2": "You are a designer named Bo. Introduce yourself to the others, then wait for their "
    "replies.",
}


async def run_team() -> str:
    hub = Hub()
    names = list(TASKS)
    for n in names:
        hub.register(n)

    helper_tasks = [
        asyncio.create_task(
            run_agent(
                hub,
                n,
                system=TEAM_SYSTEM.format(name=n, peers=[p for p in names if p != n]),
                first_user_turn=TASKS[n],
            )
        )
        for n in names[1:]
    ]
    try:
        return await run_agent(
            hub,
            "lead",
            system=TEAM_SYSTEM.format(name="lead", peers=names[1:]),
            first_user_turn=TASKS["lead"],
        )
    finally:
        for t in helper_tasks:
            t.cancel()
        await asyncio.gather(*helper_tasks, return_exceptions=True)

In [13]:
answer = await run_team()
print(f"\n[lead final answer]\n{answer}\n\nTool-call summary:")
print_trace()

  [helper1] send_message({'recipient_ids': ['lead', 'helper2'], 'content': "Hi all, I…) → delivered to ['lead', 'helper2']
  [helper2] send_message({'recipient_ids': ['lead', 'helper1'], 'content': "Hi everyo…) → delivered to ['lead', 'helper1']
  [helper2] ← received from helper1: Hi all, I'm Ada, a backend engineer on the team. I focus on …
  [lead] send_message({'recipient_ids': ['helper1', 'helper2'], 'content': "Hi tea…) → delivered to ['helper1', 'helper2']
  [lead] ← received from helper1: Hi all, I'm Ada, a backend engineer on the team. I focus on …
  [lead] ← received from helper2: Hi everyone! I'm Bo, the designer on the team. Looking forwa…
  [helper1] wait_for_message({}) → woke: new messages
  [helper1] ← received from helper2: Hi everyone! I'm Bo, the designer on the team. Looking forwa…
  [helper1] ← received from lead: Hi team, I'm lead, the coordinator for our group. Looking fo…
  [helper2] wait_for_message({}) → woke: new messages
  [helper2] ← received from lead: Hi 

---
## 2부 · 비동기 서브에이전트 (동적 생성)

이번에는 리드에 **서브에이전트 도구**와 클라이언트 측의 사소한 `sleep(seconds)` 도구를 줍니다. 리드가 헬퍼 셋을 띄우면 각자 *N*초 동안 잠들었다가 리드에 `send_message`로 "완료"를 보내고, `wait_for_message`로 다음 지시를 기다립니다. 생성 호출은 즉시 반환되므로, 리드는 이들이 도는 동안 `get_status`를 확인하고 `wait_for_message`로 보고를 모은 뒤 `kill_subagents`로 셋 모두를 정리합니다. 도메인 로직은 없고 생성 / 상태 확인 / 수집 / 정리의 전체 경로만 담았습니다.

In [15]:
SLEEP = {
    "name": "sleep",
    "description": "Sleep for the given number of seconds, then return.",
    "input_schema": {
        "type": "object",
        "properties": {"seconds": {"type": "integer", "minimum": 0, "maximum": 10}},
        "required": ["seconds"],
    },
}
SUBAGENT_TOOLS = [
    {
        "name": "create_subagents",
        "description": "Spawn helper subagents. Returns immediately — helpers run concurrently in the "
        "background in parallel with you. Each gets base_instruction, optionally + "
        "per_subagent_instructions[i].",
        "input_schema": {
            "type": "object",
            "properties": {
                "base_instruction": {"type": "string"},
                "per_subagent_instructions": {
                    "type": "array",
                    "items": {"type": "string"},
                    "maxItems": 10,
                },
            },
            "required": ["base_instruction"],
        },
    },
    {
        "name": "get_status",
        "description": "Status of every helper (active / idling / done / crashed).",
        "input_schema": {"type": "object", "properties": {}},
    },
    {
        "name": "kill_subagents",
        "description": "Cancel running helpers you no longer need.",
        "input_schema": {
            "type": "object",
            "properties": {
                "subagent_ids": {"type": "array", "items": {"type": "string"}, "minItems": 1}
            },
            "required": ["subagent_ids"],
        },
    },
]


async def dispatch_sleep(block) -> str:
    s = int(block.input["seconds"])
    await asyncio.sleep(s)
    return f"slept {s}s"

In [16]:
async def run_spawn_lead() -> str:
    hub = Hub()
    hub.register("lead")
    helpers: dict[str, asyncio.Task] = {}

    async def _create(block):
        base = block.input["base_instruction"]
        per = block.input.get("per_subagent_instructions") or [""]
        spawned = []
        for suffix in per:
            h = hub.new_name()
            helpers[h] = asyncio.create_task(
                run_agent(
                    hub,
                    h,
                    system=f"You are {h}, a helper.",
                    first_user_turn=f"{base}\n\n{suffix}".strip(),
                    tools=[SLEEP, *BASE_TOOLS],
                    extra_dispatch={"sleep": dispatch_sleep},
                )
            )
            spawned.append(h)
        return f"spawned: {', '.join(spawned)}"

    async def _status(block):
        return "\n".join(f"{n}: {s}" for n, s in hub.status.items() if n != "lead") or "(none)"

    async def _kill(block):
        killed_ids, to_await = [], []
        for sid in block.input["subagent_ids"]:
            if sid in helpers and not helpers[sid].done():
                helpers[sid].cancel()
                hub.status[sid] = "done"
                to_await.append(helpers.pop(sid))
                killed_ids.append(sid)
        await asyncio.gather(*to_await, return_exceptions=True)
        return f"cancelled: {', '.join(killed_ids)}" if killed_ids else "no matching active helpers"

    try:
        return await run_agent(
            hub,
            "lead",
            system="You are the lead.",
            first_user_turn=(
                "Spawn three helper agents. Instruct each to sleep a different number of seconds "
                "(1, 2, 3), report back to you 'done, slept Ns', then wait for your further "
                "instructions. After that, check that they're running, collect all three reports, "
                "then dismiss all three helpers. Finish with a one-line summary."
            ),
            tools=[*SUBAGENT_TOOLS, *BASE_TOOLS],
            extra_dispatch={
                "create_subagents": _create,
                "get_status": _status,
                "kill_subagents": _kill,
            },
        )
    finally:
        for t in helpers.values():
            t.cancel()
        await asyncio.gather(*helpers.values(), return_exceptions=True)

In [17]:
answer = await run_spawn_lead()
print(f"\n[lead final answer]\n{answer}\n\nTool-call summary:")
print_trace()

  [lead] create_subagents({'base_instruction': 'You are a helper agent. Follow your sp…) → spawned: helper1, helper2, helper3
  [lead] get_status({}) → helper1: active helper2: active helper3: active
  [helper1] sleep({'seconds': 1}) → slept 1s
  [helper2] sleep({'seconds': 2}) → slept 2s
  [helper1] send_message({'recipient_ids': ['lead'], 'content': 'done, slept 1s'}) → delivered to ['lead']
  [lead] wait_for_message({}) → woke: new messages
  [lead] ← received from helper1: done, slept 1s
  [helper3] sleep({'seconds': 3}) → slept 3s
  [helper2] send_message({'recipient_ids': ['lead'], 'content': 'done, slept 2s'}) → delivered to ['lead']
  [lead] wait_for_message({}) → woke: new messages
  [lead] ← received from helper2: done, slept 2s
  [helper3] send_message({'recipient_ids': ['lead'], 'content': 'done, slept 3s'}) → delivered to ['lead']
  [lead] wait_for_message({}) → woke: new messages
  [lead] ← received from helper3: done, slept 3s
  [lead] kill_subagents({'subagent_ids': ['h

---
## 다음 단계

여러분의 도메인 도구로 바꿔 넣으세요. `tools`에 추가하고 `extra_dispatch`에 핸들러를 더하면 됩니다. 위의 `Hub`, `run_agent`, 그리고 두 팀 러너면 충분하고, 나머지는 여러분의 작업과 도구입니다.

전체 도구 정의 레퍼런스는 [도구 사용 가이드](https://docs.claude.com/en/docs/tool-use)를 참고하세요.